In [1]:
import sys; sys.path.append("..")
import time
import pandas as pd

D = "../dataset/dev/"
# WHY only these two files: the raw tables are not needed and would waste RAM
train_pairs = pd.read_pickle(D + "train_pairs.pkl")
val_pairs = pd.read_pickle(D + "val_pairs.pkl")

# WHY exclude these: s1 and cand are IDs and label is the answer, none may be a feature
FEATURES = [c for c in train_pairs.columns if c not in ("s1", "cand", "label")]
print(train_pairs.shape, val_pairs.shape)
print(FEATURES)

# WHY a separate try: if lightgbm is missing on Python 3.14, we learn it before any training
import lightgbm as lgb
print("lightgbm", lgb.__version__)

(2309975, 15) (577150, 15)
['name_missing', 'name_set', 'name_sort', 'name_ratio', 'name_tri', 'name_tok', 'addr_missing', 'addr_jaccard', 'addr_min', 'num_match', 'num_conflict', 'word_jaccard']
lightgbm 4.7.0


In [4]:
import time
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

# WHY fixed settings: a plain first model, so later changes can be compared against it
model = lgb.LGBMClassifier(
    n_estimators=200,      # number of trees
    learning_rate=0.1,     # WHY 0.1: a common default, fast to converge
    num_leaves=31,         # WHY 31: moderate size, low risk of memorising training pairs
    random_state=42,       # WHY: same result on every run
    verbose=-1,
)

t0 = time.time()
model.fit(train_pairs[FEATURES], train_pairs["label"])
print("train seconds:", round(time.time() - t0, 1))

# WHY predict_proba[:, 1]: the probability of class 1 (match), which we will threshold later
val_pairs["prob"] = model.predict_proba(val_pairs[FEATURES])[:, 1]

print("val AUC:", round(roc_auc_score(val_pairs["label"], val_pairs["prob"]), 4))
print("val average precision:", round(average_precision_score(val_pairs["label"], val_pairs["prob"]), 4))

# WHY importance: shows which features the model relies on, useful for the write-up
print(dict(zip(FEATURES, model.feature_importances_)),"\n")

total = model.feature_importances_.sum()
print({f: f"{round((imp / total) * 100, 2)}%" for f, imp in zip(FEATURES, model.feature_importances_)})

train seconds: 5.6
val AUC: 0.9988
val average precision: 0.9722
{'name_missing': np.int32(63), 'name_set': np.int32(549), 'name_sort': np.int32(572), 'name_ratio': np.int32(772), 'name_tri': np.int32(837), 'name_tok': np.int32(486), 'addr_missing': np.int32(177), 'addr_jaccard': np.int32(780), 'addr_min': np.int32(818), 'num_match': np.int32(87), 'num_conflict': np.int32(167), 'word_jaccard': np.int32(692)} 

{'name_missing': '1.05%', 'name_set': '9.15%', 'name_sort': '9.53%', 'name_ratio': '12.87%', 'name_tri': '13.95%', 'name_tok': '8.1%', 'addr_missing': '2.95%', 'addr_jaccard': '13.0%', 'addr_min': '13.63%', 'num_match': '1.45%', 'num_conflict': '2.78%', 'word_jaccard': '11.53%'}
